# RNNs and LSTMs

## Learning Objectives
1. Implement a vanilla RNN forward pass in NumPy to understand hidden state recurrence and tanh gating.
2. Build a bidirectional LSTM classifier in PyTorch with packed sequences and OOM-safe batching.
3. Apply an LSTM to sentiment analysis showing real-world NLP preprocessing and evaluation.
4. Compare LSTM vs Transformer on long sequences to understand when recurrence still wins.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: Vanilla RNN Forward Pass — NumPy

In [ ]:
# Manual RNN forward: h_t = tanh(W_h @ h_{t-1} + W_x @ x_t + b)
# Shows how information flows through time via the hidden state.

class VanillaRNN:
    """Single-layer RNN implemented in NumPy."""

    def __init__(self, input_size: int, hidden_size: int):
        self.hidden_size = hidden_size
        scale = 0.01
        rng = np.random.default_rng(42)
        # Weight matrices
        self.W_h = rng.standard_normal((hidden_size, hidden_size)) * scale
        self.W_x = rng.standard_normal((hidden_size, input_size)) * scale
        self.b = np.zeros(hidden_size)

    def forward(self, x: np.ndarray) -> tuple:
        """Process sequence x of shape (seq_len, input_size).
        
        Returns (hidden_states, final_hidden) where hidden_states has
        shape (seq_len, hidden_size).
        """
        seq_len = x.shape[0]
        h = np.zeros(self.hidden_size)  # Initial hidden state
        hidden_states = []
        for t in range(seq_len):
            # h_t = tanh(W_h @ h_{t-1} + W_x @ x_t + b)
            h = np.tanh(self.W_h @ h + self.W_x @ x[t] + self.b)
            hidden_states.append(h.copy())
        return np.array(hidden_states), h


rnn = VanillaRNN(input_size=10, hidden_size=16)

# Simulate a sequence of length 20 (e.g., one-hot embeddings)
seq = np.random.randn(20, 10)
hidden_states, final_h = rnn.forward(seq)
print(f"Input sequence shape: {seq.shape}")
print(f"Hidden states shape:  {hidden_states.shape}")
print(f"Final hidden state:   {final_h.shape}")
print(f"Hidden state norm evolution (first 5 steps):")
for t in range(5):
    print(f"  t={t}: ||h_t|| = {np.linalg.norm(hidden_states[t]):.4f}")
print(f"  ... mean norm over all steps: {np.linalg.norm(hidden_states, axis=1).mean():.4f}")

## Level 2: Bidirectional LSTM Classifier — PyTorch with OOM Handling

In [ ]:
# BiLSTM: processes sequence left-to-right AND right-to-left.
# Concatenates both final hidden states for classification.
# Uses packed sequences for variable-length inputs (production pattern).

class BiLSTMClassifier(nn.Module):
    """Bidirectional LSTM with pooling for sequence classification."""

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 64,
        hidden_size: int = 128,
        n_layers: int = 2,
        n_classes: int = 2,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_size,
            num_layers=n_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        # Bidirectional doubles the output size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, n_classes),
        )

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len) token ids; lengths: (batch,) actual lengths."""
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        # Pack for variable-length efficiency
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        output, (h_n, _) = self.lstm(packed)
        # h_n: (n_layers*2, batch, hidden_size) — take last layer both directions
        # Forward: h_n[-2], Backward: h_n[-1]
        h_fwd = h_n[-2]   # (batch, hidden_size)
        h_bwd = h_n[-1]   # (batch, hidden_size)
        h_cat = torch.cat([h_fwd, h_bwd], dim=-1)  # (batch, hidden*2)
        return self.classifier(h_cat)


# --- Synthetic sentiment dataset ---
VOCAB = 500
MAX_LEN = 50
N = 1000

rng_t = torch.Generator().manual_seed(42)
tokens = torch.randint(1, VOCAB, (N, MAX_LEN), generator=rng_t)
# Randomly pad some sequences to simulate variable length
lengths = torch.randint(10, MAX_LEN, (N,), generator=rng_t)
for i, l in enumerate(lengths):
    tokens[i, l:] = 0  # Padding token
labels = torch.randint(0, 2, (N,), generator=rng_t)

split = 800
tr_loader = DataLoader(
    TensorDataset(tokens[:split], lengths[:split], labels[:split]),
    batch_size=64, shuffle=True
)
vl_loader = DataLoader(
    TensorDataset(tokens[split:], lengths[split:], labels[split:]),
    batch_size=64
)

model = BiLSTMClassifier(vocab_size=VOCAB).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    for xb, lb, yb in tr_loader:
        xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
        try:
            opt.zero_grad()
            crit(model(xb, lb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Prevent exploding gradients
            opt.step()
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM — reduce batch size or sequence length")
                torch.cuda.empty_cache()
                continue
            raise

model.eval()
correct = total = 0
with torch.no_grad():
    for xb, lb, yb in vl_loader:
        xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
        correct += (model(xb, lb).argmax(1) == yb).sum().item()
        total += len(yb)
print(f"BiLSTM val accuracy: {correct/total:.3f}")

## Real-World Example 1: Sentiment Analysis with LSTM

In [ ]:
# Production sentiment analysis pipeline:
# tokenize -> pad -> embed -> LSTM -> classify
# Shows full preprocessing + training + evaluation loop.

# Simulate reviews with vocabulary statistics matching real-world distributions
def generate_reviews(
    n: int, vocab_size: int = 1000, max_len: int = 60, seed: int = 0
) -> tuple:
    """Generate (tokens, lengths, labels) synthetic review data."""
    rng = np.random.default_rng(seed)
    # Positive reviews use high token IDs, negative reviews use low token IDs
    labels = (rng.random(n) > 0.5).astype(int)
    tokens = np.zeros((n, max_len), dtype=int)
    lengths = rng.integers(15, max_len, n)
    for i, (label, length) in enumerate(zip(labels, lengths)):
        if label == 1:  # Positive
            tokens[i, :length] = rng.integers(vocab_size // 2, vocab_size, length)
        else:  # Negative
            tokens[i, :length] = rng.integers(1, vocab_size // 2, length)
    return (
        torch.tensor(tokens, dtype=torch.long),
        torch.tensor(lengths, dtype=torch.long),
        torch.tensor(labels, dtype=torch.long),
    )


VOCAB = 1000
tok_tr, len_tr, lbl_tr = generate_reviews(1200, VOCAB, seed=0)
tok_vl, len_vl, lbl_vl = generate_reviews(300, VOCAB, seed=1)

sa_loader_tr = DataLoader(
    TensorDataset(tok_tr, len_tr, lbl_tr), batch_size=64, shuffle=True
)
sa_loader_vl = DataLoader(
    TensorDataset(tok_vl, len_vl, lbl_vl), batch_size=64
)

sa_model = BiLSTMClassifier(vocab_size=VOCAB, embed_dim=64, hidden_size=64, n_layers=1).to(device)
sa_opt = torch.optim.Adam(sa_model.parameters(), lr=2e-3)
sa_crit = nn.CrossEntropyLoss()

best_acc = 0.0
for epoch in range(15):
    sa_model.train()
    for xb, lb, yb in sa_loader_tr:
        xb, lb, yb = xb.to(device), lb, yb.to(device)
        sa_opt.zero_grad()
        sa_crit(sa_model(xb, lb), yb).backward()
        nn.utils.clip_grad_norm_(sa_model.parameters(), 1.0)
        sa_opt.step()

    sa_model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, lb, yb in sa_loader_vl:
            correct += (sa_model(xb.to(device), lb).argmax(1) == yb.to(device)).sum().item()
            total += len(yb)
    acc = correct / total
    best_acc = max(best_acc, acc)

print(f"Sentiment LSTM | best val accuracy: {best_acc:.3f}")
print(f"In production: use pretrained GloVe/fastText embeddings -> embed_dim 100-300")

## Real-World Example 2: Time Series LSTM Forecasting

In [ ]:
# LSTM for multi-step time series forecasting.
# Sliding window: given last window_size steps, predict next step.

class LSTMForecaster(nn.Module):
    """LSTM-based sequence-to-one forecaster."""

    def __init__(self, input_size: int = 1, hidden_size: int = 64, n_layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, n_layers, batch_first=True, dropout=0.1
        )
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, window_size, 1)"""
        out, _ = self.lstm(x)  # (batch, window_size, hidden_size)
        return self.linear(out[:, -1, :])  # Predict from last timestep


# Synthetic sine wave + noise
t = np.linspace(0, 40 * np.pi, 5000)
signal = np.sin(t) + 0.1 * np.random.randn(5000)
signal = (signal - signal.mean()) / signal.std()  # Normalize

WINDOW = 50
# Create sliding windows
X_ts = np.array([signal[i:i+WINDOW] for i in range(len(signal) - WINDOW - 1)])
y_ts = signal[WINDOW+1:]
# Convert to tensors (batch, window, 1)
X_ts_t = torch.tensor(X_ts, dtype=torch.float32).unsqueeze(-1)
y_ts_t = torch.tensor(y_ts, dtype=torch.float32).unsqueeze(-1)

split = int(0.8 * len(X_ts_t))
ts_tr = DataLoader(TensorDataset(X_ts_t[:split], y_ts_t[:split]), batch_size=128, shuffle=True)
ts_vl = DataLoader(TensorDataset(X_ts_t[split:], y_ts_t[split:]), batch_size=128)

forecaster = LSTMForecaster().to(device)
ts_opt = torch.optim.Adam(forecaster.parameters(), lr=1e-3)
ts_crit = nn.MSELoss()

for epoch in range(15):
    forecaster.train()
    for xb, yb in ts_tr:
        xb, yb = xb.to(device), yb.to(device)
        ts_opt.zero_grad()
        ts_crit(forecaster(xb), yb).backward()
        ts_opt.step()

forecaster.eval()
val_mse = 0.0
with torch.no_grad():
    for xb, yb in ts_vl:
        xb, yb = xb.to(device), yb.to(device)
        val_mse += ts_crit(forecaster(xb), yb).item() * len(xb)
val_mse /= len(ts_vl.dataset)
print(f"Forecasting LSTM | val MSE: {val_mse:.6f}")
print(f"Baseline (predict mean): {signal[WINDOW+1:].var():.6f}")

## Real-World Example 3: LSTM vs Transformer on Long Sequences

In [ ]:
# Benchmark LSTM vs small Transformer on sequences of increasing length.
# LSTM: O(T) memory, O(T) compute (sequential)
# Transformer: O(T^2) memory from attention, but parallelizable.

import time

class SmallTransformerClassifier(nn.Module):
    """Minimal Transformer encoder for sequence classification."""

    def __init__(self, vocab: int, d_model: int = 64, nhead: int = 4, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model, padding_idx=0)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=128, batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.clf = nn.Linear(d_model, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.encoder(self.embed(x))
        return self.clf(out.mean(dim=1))  # Mean pooling


class SimpleLSTMClassifier(nn.Module):
    def __init__(self, vocab: int, d_model: int = 64, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model, padding_idx=0)
        self.lstm = nn.LSTM(d_model, d_model, batch_first=True)
        self.clf = nn.Linear(d_model, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (h_n, _) = self.lstm(self.embed(x))
        return self.clf(h_n[-1])


VOCAB = 200
BATCH = 32

print(f"{'Seq Len':>10} | {'LSTM (ms)':>12} | {'Transformer (ms)':>17}")
print("-" * 48)
for seq_len in [32, 64, 128, 256, 512]:
    x_dummy = torch.randint(1, VOCAB, (BATCH, seq_len), device=device)
    lstm_model = SimpleLSTMClassifier(VOCAB).to(device).eval()
    tfm_model = SmallTransformerClassifier(VOCAB).to(device).eval()

    with torch.no_grad():
        # Warmup
        lstm_model(x_dummy); tfm_model(x_dummy)
        # Time LSTM
        t0 = time.perf_counter()
        for _ in range(50): lstm_model(x_dummy)
        lat_lstm = (time.perf_counter() - t0) / 50 * 1000
        # Time Transformer
        t0 = time.perf_counter()
        for _ in range(50): tfm_model(x_dummy)
        lat_tfm = (time.perf_counter() - t0) / 50 * 1000

    print(f"{seq_len:>10} | {lat_lstm:12.2f} | {lat_tfm:17.2f}")

print("\nGuidance:")
print("  LSTM: better for very long sequences (>1K) due to O(T) memory.")
print("  Transformer: better at parallelism on GPU for medium lengths (<512).")